In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

flights = pd.read_parquet("dataset/merged_flights.parquet")
print(f"Loaded: {flights.shape}")
print(f"Columns: {flights.columns.tolist()}")

Loaded: (18227796, 83)
Columns: ['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_AIRLINE_ID', 'TAIL_NUM', 'OP_CARRIER_FL_NUM', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'ORIGIN_STATE_NM', 'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR', 'DEST_STATE_NM', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15', 'DEP_TIME_BLK', 'TAXI_OUT', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15', 'ARR_TIME_BLK', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME', 'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'FLIGHTS', 'DISTANCE', 'DISTANCE_GROUP', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'AIRLINE_NAME', 'DEP_HOUR', 'DAY_NAME', 'DIST_GROUP', 'TAXI_GROUP', 'AIRLINE', 'IS_HOLIDAY', 'ARR_HOUR', 'FL_DATE_DATE', 'origin_temp', 'origin_dew_point', 'origin_pressure', 'origin_wind_dir', 'origin_wind_speed', 'origin_sky_condition', 'origin_precip_1hr', '

In [3]:
flights.CANCELLATION_CODE.nunique()

0

In [4]:
flights.CANCELLED.value_counts()

CANCELLED
0.0    18227796
Name: count, dtype: int64

In [5]:
flights.DIVERTED.value_counts()

DIVERTED
0.0    18227796
Name: count, dtype: int64

In [6]:
flights.FLIGHTS.value_counts()

FLIGHTS
1.0    18227796
Name: count, dtype: int64

In [ ]:
print("── CANCELLATION_CODE ────────────────────────────────────────")
print(flights["CANCELLATION_CODE"].value_counts(dropna=False))

print("\n── CANCELLED ────────────────────────────────────────────────")
print(flights["CANCELLED"].value_counts(dropna=False))

print("\n── DIVERTED ─────────────────────────────────────────────────")
print(flights["DIVERTED"].value_counts(dropna=False))

print("\n── FLIGHTS ──────────────────────────────────────────────────")
print(flights["FLIGHTS"].value_counts(dropna=False))

print("\n── DISTANCE_GROUP vs DIST_GROUP ─────────────────────────────")
print("DISTANCE_GROUP:", sorted(flights["DISTANCE_GROUP"].unique().tolist()))
print("DIST_GROUP:", sorted(flights["DIST_GROUP"].dropna().unique().tolist()))

print("\n── AIRLINE vs AIRLINE_NAME ──────────────────────────────────")
print(flights[["AIRLINE", "AIRLINE_NAME"]].drop_duplicates().sort_values("AIRLINE"))

print("\n── FL_DATE_DATE sample ──────────────────────────────────────")
print(flights["FL_DATE_DATE"].head(5))
print("Redundant with FL_DATE?", (pd.to_datetime(flights["FL_DATE_DATE"]) == flights["FL_DATE"]).all())

print("\n── DEP_TIME_BLK sample ──────────────────────────────────────")
print(flights["DEP_TIME_BLK"].value_counts().head(8))

print("\n── ARR_TIME_BLK sample ──────────────────────────────────────")
print(flights["ARR_TIME_BLK"].value_counts().head(8))

print("\n── origin_precip_6hr null rate ──────────────────────────────")
print(f"  {flights['origin_precip_6hr'].isna().mean()*100:.1f}% null")
print(f"  Non-null sample: {flights['origin_precip_6hr'].dropna().head(5).tolist()}")

print("\n── dest_precip_6hr null rate ────────────────────────────────")
print(f"  {flights['dest_precip_6hr'].isna().mean()*100:.1f}% null")
print(f"  Non-null sample: {flights['dest_precip_6hr'].dropna().head(5).tolist()}")

── CANCELLATION_CODE ────────────────────────────────────────
CANCELLATION_CODE
None    18227796
Name: count, dtype: int64

── CANCELLED ────────────────────────────────────────────────
CANCELLED
0.0    18227796
Name: count, dtype: int64

── DIVERTED ─────────────────────────────────────────────────
DIVERTED
0.0    18227796
Name: count, dtype: int64

── FLIGHTS ──────────────────────────────────────────────────
FLIGHTS
1.0    18227796
Name: count, dtype: int64

── DISTANCE_GROUP vs DIST_GROUP ─────────────────────────────
DISTANCE_GROUP: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
DIST_GROUP: ['0-500', '1.5K-2K', '1K-1.5K', '2K-3K', '3K+', '500-1K']

── AIRLINE vs AIRLINE_NAME ──────────────────────────────────
            AIRLINE       AIRLINE_NAME
2577         Alaska    Alaska Airlines
6106      Allegiant          Allegiant
348        American  American Airlines
3863          Delta              Delta
0          Endeavor       Endeavor Air
6544      Envoy Air          Envoy Air
5694       Fro

In [ ]:
DROP_COLS = [
    "CANCELLATION_CODE",   # 100% null
    "origin_precip_6hr",   # 97% null, unreliable
    "dest_precip_6hr",     # 97% null, unreliable
    "CANCELLED",           # all 0 — removed in Step 1
    "DIVERTED",            # all 0 — removed in Step 1
    "FLIGHTS",             # constant 1 — useless
    "DISTANCE_GROUP",      # duplicate of DIST_GROUP
    "AIRLINE",             # duplicate of AIRLINE_NAME
    "FL_DATE_DATE",        # join artifact, redundant with FL_DATE
    "DEP_TIME_BLK",        # superseded by DEP_HOUR
    "ARR_TIME_BLK",        # superseded by ARR_HOUR
]

flights.drop(columns=DROP_COLS, inplace=True)
print(f"Dropped {len(DROP_COLS)} cols. New shape: {flights.shape}")
print(f"Remaining cols: {flights.columns.tolist()}")

Dropped 11 cols. New shape: (18227796, 72)
Remaining cols: ['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_AIRLINE_ID', 'TAIL_NUM', 'OP_CARRIER_FL_NUM', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'ORIGIN_STATE_NM', 'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR', 'DEST_STATE_NM', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15', 'TAXI_OUT', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15', 'CRS_ELAPSED_TIME', 'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'AIRLINE_NAME', 'DEP_HOUR', 'DAY_NAME', 'DIST_GROUP', 'TAXI_GROUP', 'IS_HOLIDAY', 'ARR_HOUR', 'origin_temp', 'origin_dew_point', 'origin_pressure', 'origin_wind_dir', 'origin_wind_speed', 'origin_sky_condition', 'origin_precip_1hr', 'dest_temp', 'dest_dew_point', 'dest_pressure', 'dest_wind_dir', 'dest_wind_speed', 'dest_sky_condition', 'd

In [9]:
pd.set_option("Display.Max_columns",None)
flights.head()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_NM,DEST,DEST_CITY_NAME,DEST_STATE_ABR,DEST_STATE_NM,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,TAXI_OUT,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,AIRLINE_NAME,DEP_HOUR,DAY_NAME,DIST_GROUP,TAXI_GROUP,IS_HOLIDAY,ARR_HOUR,origin_temp,origin_dew_point,origin_pressure,origin_wind_dir,origin_wind_speed,origin_sky_condition,origin_precip_1hr,dest_temp,dest_dew_point,dest_pressure,dest_wind_dir,dest_wind_speed,dest_sky_condition,dest_precip_1hr,OP_REVENUES,OP_EXPENSES,NET_INCOME,OP_PROFIT_LOSS,FLYING_OPS,MAINTENANCE,PAX_SERVICE,TRANS_REV_PAX,DEPREC_AMORT,TRANS_EXPENSES,origin_monthly_passengers,origin_monthly_departures
0,2023,1,1,1,7,2023-01-01,9E,20363,N131EV,5244.0,ORD,"Chicago, IL",IL,Illinois,JFK,"New York, NY",NY,New York,1520,1524.0,4.0,4.0,0.0,20.0,23.0,1841,1838.0,-3.0,0.0,0.0,141.0,134.0,91.0,740.0,NaN,NaN,NaN,NaN,NaN,Endeavor Air,15,Sun,500-1K,10-20,1,18,5.6,3.9,1012.8,260.0,2.6,NaN,0.0,12.8,3.9,1013.7,290.0,6.7,4.0,0.0,143826.04,233156.8,-89175.51,-89330.76,116882.94,69991.6,25028.93,143826.04,3109.96,0.0,1855892.0,416.0
1,2023,1,1,1,7,2023-01-01,9E,20363,N131EV,5317.0,JFK,"New York, NY",NY,New York,ORD,"Chicago, IL",IL,Illinois,945,941.0,-4.0,0.0,0.0,25.0,8.0,1144,1120.0,-24.0,0.0,0.0,179.0,159.0,126.0,740.0,NaN,NaN,NaN,NaN,NaN,Endeavor Air,9,Sun,500-1K,20-30,1,11,8.9,8.9,1008.6,250.0,5.1,6.0,0.0,3.3,2.2,1010.3,140.0,2.1,NaN,-0.1,143826.04,233156.8,-89175.51,-89330.76,116882.94,69991.6,25028.93,143826.04,3109.96,0.0,1075492.0,160.0
2,2023,1,1,1,7,2023-01-01,9E,20363,N131EV,5397.0,JFK,"New York, NY",NY,New York,BGR,"Bangor, ME",ME,Maine,2100,2056.0,-4.0,0.0,0.0,28.0,5.0,2236,2229.0,-7.0,0.0,0.0,96.0,93.0,60.0,382.0,NaN,NaN,NaN,NaN,NaN,Endeavor Air,21,Sun,0-500,20-30,1,22,12.2,0.6,1014.2,300.0,4.6,4.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,143826.04,233156.8,-89175.51,-89330.76,116882.94,69991.6,25028.93,143826.04,3109.96,0.0,1075492.0,160.0
3,2023,1,1,1,7,2023-01-01,9E,20363,N133EV,5076.0,ATL,"Atlanta, GA",GA,Georgia,SGF,"Springfield, MO",MO,Missouri,1130,1125.0,-5.0,0.0,0.0,17.0,4.0,1225,1214.0,-11.0,0.0,0.0,115.0,109.0,88.0,563.0,NaN,NaN,NaN,NaN,NaN,Endeavor Air,11,Sun,500-1K,10-20,1,12,12.2,11.7,1017.8,0.0,0.0,9.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,143826.04,233156.8,-89175.51,-89330.76,116882.94,69991.6,25028.93,143826.04,3109.96,0.0,3194785.0,358.0
4,2023,1,1,1,7,2023-01-01,9E,20363,N133EV,5076.0,SGF,"Springfield, MO",MO,Missouri,ATL,"Atlanta, GA",GA,Georgia,1400,1354.0,-6.0,0.0,0.0,16.0,5.0,1637,1630.0,-7.0,0.0,0.0,97.0,96.0,75.0,563.0,NaN,NaN,NaN,NaN,NaN,Endeavor Air,14,Sun,500-1K,10-20,1,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.9,13.3,1020.9,260.0,3.1,NaN,0.0,143826.04,233156.8,-89175.51,-89330.76,116882.94,69991.6,25028.93,143826.04,3109.96,0.0,32775.0,13.0


In [10]:
TARGET = ['ARR_DEL15', 'ARR_DELAY']

LEAKAGE_COLS = [
    'DEP_TIME', 'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15',
    'ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_NEW',
    'TAXI_OUT', 'TAXI_IN',
    'ACTUAL_ELAPSED_TIME', 'AIR_TIME',
    'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY',
    'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY',
    'TAXI_GROUP',
]

IDENTIFIER_COLS = [
    'FL_DATE', 'YEAR', 'QUARTER', 'MONTH',
    'OP_UNIQUE_CARRIER', 'OP_CARRIER_AIRLINE_ID',
    'TAIL_NUM', 'OP_CARRIER_FL_NUM',
    'ORIGIN', 'DEST',
    'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'ORIGIN_STATE_NM',
    'DEST_CITY_NAME', 'DEST_STATE_ABR', 'DEST_STATE_NM',
    'AIRLINE_NAME',
    'CRS_DEP_TIME', 'CRS_ARR_TIME', 'CRS_ELAPSED_TIME',
]

READY_FEATURES = [
    'DAY_OF_MONTH', 'DAY_OF_WEEK', 'DEP_HOUR', 'ARR_HOUR',
    'DAY_NAME', 'IS_HOLIDAY',
    'DISTANCE', 'DIST_GROUP',
    'origin_temp', 'origin_dew_point', 'origin_pressure',
    'origin_wind_dir', 'origin_wind_speed', 'origin_sky_condition', 'origin_precip_1hr',
    'dest_temp', 'dest_dew_point', 'dest_pressure',
    'dest_wind_dir', 'dest_wind_speed', 'dest_sky_condition', 'dest_precip_1hr',
    'OP_REVENUES', 'OP_EXPENSES', 'NET_INCOME', 'OP_PROFIT_LOSS',
    'FLYING_OPS', 'MAINTENANCE', 'PAX_SERVICE', 'TRANS_REV_PAX',
    'DEPREC_AMORT', 'TRANS_EXPENSES',
    'origin_monthly_passengers', 'origin_monthly_departures',
]

TO_ENGINEER = [
    'OP_UNIQUE_CARRIER',  # → rolling 30d airline delay rate
    'ORIGIN',             # → rolling 30d airport delay rate
    'DEST',               # → rolling 30d dest delay rate
    'TAIL_NUM',           # → cascade score
    'FL_DATE',            # → rolling window cutoffs
]

all_classified = set(TARGET + LEAKAGE_COLS + IDENTIFIER_COLS + READY_FEATURES + TO_ENGINEER)
unclassified = [c for c in flights.columns if c not in all_classified]

print(f"TARGET:          {len(TARGET)}")
print(f"LEAKAGE:         {len(LEAKAGE_COLS)}")
print(f"IDENTIFIERS:     {len(IDENTIFIER_COLS)}")
print(f"READY FEATURES:  {len(READY_FEATURES)}")
print(f"TO ENGINEER:     {len(TO_ENGINEER)}")
print(f"TOTAL IN DF:     {flights.shape[1]}")
print(f"\nUNCLASSIFIED (must be 0): {unclassified}")

TARGET:          2
LEAKAGE:         17
IDENTIFIERS:     20
READY FEATURES:  34
TO ENGINEER:     5
TOTAL IN DF:     72

UNCLASSIFIED (must be 0): []


In [11]:
flights.ARR_DEL15.value_counts()

ARR_DEL15
0.0    14353289
1.0     3874507
Name: count, dtype: int64

In [12]:
(flights.ARR_DELAY >=15 ).sum()

np.int64(3874507)

In [13]:
hourly = flights.groupby('DEP_HOUR')['ARR_DEL15'].mean() * 100
print(hourly.round(2))

DEP_HOUR
0     19.80
1     21.93
2     21.79
3     21.66
4     19.22
5      8.99
6      9.90
7     12.52
8     14.05
9     15.39
10    16.47
11    18.10
12    19.63
13    21.81
14    24.02
15    25.78
16    27.69
17    28.81
18    29.74
19    30.57
20    30.82
21    28.85
22    28.13
23    22.22
Name: ARR_DEL15, dtype: float64


In [14]:
for hour in range(14, 24):
    rate = hourly[hour]
    print(f"Hour {hour}: {rate:.2f}%")

Hour 14: 24.02%
Hour 15: 25.78%
Hour 16: 27.69%
Hour 17: 28.81%
Hour 18: 29.74%
Hour 19: 30.57%
Hour 20: 30.82%
Hour 21: 28.85%
Hour 22: 28.13%
Hour 23: 22.22%


In [15]:
flights['is_early_departure'] = flights['DEP_HOUR'].between(5, 9).astype('int8')
flights['is_peak_hours']      = flights['DEP_HOUR'].between(15, 22).astype('int8')

print("Delay rate by is_early_departure:")
print(flights.groupby('is_early_departure')['ARR_DEL15'].mean() * 100)

print("\nDelay rate by is_peak_hours:")
print(flights.groupby('is_peak_hours')['ARR_DEL15'].mean() * 100)

Delay rate by is_early_departure:
is_early_departure
0    24.908395
1    12.444758
Name: ARR_DEL15, dtype: float64

Delay rate by is_peak_hours:
is_peak_hours
0    16.338441
1    28.785095
Name: ARR_DEL15, dtype: float64


In [16]:
for hour in list(range(0, 5)) + [23]:
    rate = hourly[hour]
    print(f"Hour {hour}: {rate:.2f}%")

Hour 0: 19.80%
Hour 1: 21.93%
Hour 2: 21.79%
Hour 3: 21.66%
Hour 4: 19.22%
Hour 23: 22.22%


In [17]:
print(flights.groupby('DAY_OF_WEEK')['ARR_DEL15'].mean() * 100)

DAY_OF_WEEK
1    21.388671
2    18.011164
3    19.408721
4    22.433696
5    23.157123
6    20.468342
7    23.487295
Name: ARR_DEL15, dtype: float64


In [18]:
print(flights.groupby('DAY_OF_WEEK')['ARR_DEL15'].mean() * 100)
print("\nDAY_OF_WEEK mapping:")
print("1=Mon, 2=Tue, 3=Wed, 4=Thu, 5=Fri, 6=Sat, 7=Sun")

DAY_OF_WEEK
1    21.388671
2    18.011164
3    19.408721
4    22.433696
5    23.157123
6    20.468342
7    23.487295
Name: ARR_DEL15, dtype: float64

DAY_OF_WEEK mapping:
1=Mon, 2=Tue, 3=Wed, 4=Thu, 5=Fri, 6=Sat, 7=Sun


In [19]:
flights.DAY_OF_WEEK.dtype

dtype('int64')

In [20]:
print(flights[['DAY_OF_WEEK', 'DAY_NAME']].drop_duplicates().sort_values('DAY_OF_WEEK'))

       DAY_OF_WEEK DAY_NAME
15470            1      Mon
32570            2      Tue
49432            3      Wed
65520            4      Thu
82293            5      Fri
99257            6      Sat
0                7      Sun


In [21]:
flights['is_high_delay_day'] = flights['DAY_OF_WEEK'].isin([4, 5, 7]).astype('int8')
flights['is_low_delay_day']  = flights['DAY_OF_WEEK'].isin([2, 3]).astype('int8')

print("Delay rate by is_high_delay_day:")
print(flights.groupby('is_high_delay_day')['ARR_DEL15'].mean() * 100)

print("\nDelay rate by is_low_delay_day:")
print(flights.groupby('is_low_delay_day')['ARR_DEL15'].mean() * 100)

Delay rate by is_high_delay_day:
is_high_delay_day
0    19.842454
1    23.025406
Name: ARR_DEL15, dtype: float64

Delay rate by is_low_delay_day:
is_low_delay_day
0    22.229354
1    18.714910
Name: ARR_DEL15, dtype: float64


In [22]:
print(flights.groupby('MONTH')['ARR_DEL15'].mean() * 100)

MONTH
1     21.669113
2     18.485110
3     21.386917
4     20.496048
5     22.764157
6     26.742567
7     29.029416
8     22.700994
9     17.063352
10    14.389834
11    14.126907
12    18.482288
Name: ARR_DEL15, dtype: float64


In [23]:
print(flights[['MONTH', 'FL_DATE']].drop_duplicates().sort_values('MONTH').groupby('MONTH').first())

         FL_DATE
MONTH           
1     2023-01-01
2     2023-02-16
3     2024-03-16
4     2025-04-14
5     2024-05-23
6     2024-06-08
7     2023-07-04
8     2025-08-09
9     2023-09-13
10    2024-10-03
11    2023-11-21
12    2024-12-29


In [24]:
flights['is_summer_peak'] = flights['MONTH'].isin([6, 7]).astype('int8')
flights['is_fall_low']    = flights['MONTH'].isin([10, 11]).astype('int8')

print("Delay rate by is_summer_peak:")
print(flights.groupby('is_summer_peak')['ARR_DEL15'].mean() * 100)

print("\nDelay rate by is_fall_low:")
print(flights.groupby('is_fall_low')['ARR_DEL15'].mean() * 100)

Delay rate by is_summer_peak:
is_summer_peak
0    19.634490
1    27.901425
Name: ARR_DEL15, dtype: float64

Delay rate by is_fall_low:
is_fall_low
0    22.285518
1    14.262315
Name: ARR_DEL15, dtype: float64


In [25]:
print(flights.columns.tolist())

['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_AIRLINE_ID', 'TAIL_NUM', 'OP_CARRIER_FL_NUM', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'ORIGIN_STATE_NM', 'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR', 'DEST_STATE_NM', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15', 'TAXI_OUT', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15', 'CRS_ELAPSED_TIME', 'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'AIRLINE_NAME', 'DEP_HOUR', 'DAY_NAME', 'DIST_GROUP', 'TAXI_GROUP', 'IS_HOLIDAY', 'ARR_HOUR', 'origin_temp', 'origin_dew_point', 'origin_pressure', 'origin_wind_dir', 'origin_wind_speed', 'origin_sky_condition', 'origin_precip_1hr', 'dest_temp', 'dest_dew_point', 'dest_pressure', 'dest_wind_dir', 'dest_wind_speed', 'dest_sky_condition', 'dest_precip_1hr', 'OP_REVENUES', 'OP_EXPENSES', 'NET_INCOME'

In [26]:
fin_cols = ['OP_REVENUES', 'OP_EXPENSES', 'NET_INCOME', 'OP_PROFIT_LOSS',
            'FLYING_OPS', 'MAINTENANCE', 'PAX_SERVICE', 'TRANS_REV_PAX',
            'DEPREC_AMORT', 'TRANS_EXPENSES']

print(flights[fin_cols].describe().round(2))

       OP_REVENUES  OP_EXPENSES   NET_INCOME  OP_PROFIT_LOSS   FLYING_OPS  \
count  18227796.00  18227796.00  18227796.00     18227796.00  18227796.00   
mean    5460884.20   5151253.82    203030.84       309630.38   1727954.14   
std     4039855.69   3699891.86    388408.94       503969.31   1137701.47   
min      135703.80    213405.96   -539748.67      -500721.12     93494.69   
25%      774843.26    724110.28    -20255.87        -6653.40    291563.54   
50%     6685060.00   6503009.00     52221.00        82518.04   2448100.11   
75%     9086365.85   8484490.44    372655.68       703800.00   2713187.00   
max    11686101.00  10715710.00   1550496.00      1795694.90   3086353.00   

       MAINTENANCE  PAX_SERVICE  TRANS_REV_PAX  DEPREC_AMORT  TRANS_EXPENSES  
count  18227796.00  18227796.00    18227796.00   18227796.00     18227796.00  
mean     510021.54    409957.93     3947284.12     243032.14       677650.05  
std      329025.70    303754.89     2747417.56     169845.08       84

In [27]:
flights['profit_margin']    = flights['OP_PROFIT_LOSS'] / flights['OP_REVENUES']
flights['maintenance_ratio'] = flights['MAINTENANCE'] / flights['OP_EXPENSES']
flights['revenue_per_pax']  = flights['TRANS_REV_PAX'] / flights['OP_REVENUES']
flights['cost_efficiency']  = flights['OP_EXPENSES'] / flights['OP_REVENUES']

print(flights[['profit_margin','maintenance_ratio',
               'revenue_per_pax','cost_efficiency']].describe().round(4))

       profit_margin  maintenance_ratio  revenue_per_pax  cost_efficiency
count   1.822780e+07       1.822780e+07     1.822780e+07     1.822780e+07
mean    1.640000e-02       1.447000e-01     7.817000e-01     9.836000e-01
std     1.294000e-01       9.640000e-02     1.371000e-01     1.294000e-01
min    -1.018600e+00       5.270000e-02     4.433000e-01     8.130000e-01
25%    -1.820000e-02       8.520000e-02     6.688000e-01     9.051000e-01
50%     4.010000e-02       9.700000e-02     7.277000e-01     9.599000e-01
75%     9.490000e-02       1.259000e-01     8.725000e-01     1.018200e+00
max     1.870000e-01       3.936000e-01     1.000000e+00     2.018600e+00


In [28]:
print(flights.groupby('AIRLINE_NAME')[['profit_margin','maintenance_ratio',
                                        'revenue_per_pax','cost_efficiency']].mean().round(4))

                   profit_margin  maintenance_ratio  revenue_per_pax  \
AIRLINE_NAME                                                           
Alaska Airlines           0.0528             0.0768           0.6997   
Allegiant                 0.0695             0.1034           0.7015   
American Airlines         0.0668             0.1124           0.6878   
Delta                     0.0880             0.0820           0.6278   
Endeavor Air             -0.5678             0.3304           1.0000   
Envoy Air                -0.0002             0.2039           0.9994   
Frontier                 -0.0068             0.0836           0.6233   
Hawaiian                 -0.0617             0.1309           0.8676   
JetBlue                  -0.0992             0.1051           0.8561   
PSA Airlines             -0.0880             0.3136           0.9999   
Republic Airways          0.1048             0.3125           0.9794   
SkyWest                   0.0229             0.3367           0.

In [29]:
regional = ['9E', 'MQ', 'OH', 'YX', 'OO']
flights['is_regional'] = flights['OP_UNIQUE_CARRIER'].isin(regional).astype('int8')

print("Delay rate by is_regional:")
print(flights.groupby('is_regional')['ARR_DEL15'].mean() * 100)

print("\nFlight counts:")
print(flights.groupby('is_regional')['ARR_DEL15'].count())

Delay rate by is_regional:
is_regional
0    22.135004
1    18.485992
Name: ARR_DEL15, dtype: float64

Flight counts:
is_regional
0    13837120
1     4390676
Name: ARR_DEL15, dtype: int64


In [30]:
print(flights[flights['is_regional']==1][['OP_UNIQUE_CARRIER','AIRLINE_NAME']].drop_duplicates().sort_values('OP_UNIQUE_CARRIER'))
print("\nNon-regional:")
print(flights[flights['is_regional']==0][['OP_UNIQUE_CARRIER','AIRLINE_NAME']].drop_duplicates().sort_values('OP_UNIQUE_CARRIER'))

      OP_UNIQUE_CARRIER      AIRLINE_NAME
0                    9E      Endeavor Air
6544                 MQ         Envoy Air
7682                 OH      PSA Airlines
8067                 OO           SkyWest
15022                YX  Republic Airways

Non-regional:
      OP_UNIQUE_CARRIER       AIRLINE_NAME
348                  AA  American Airlines
2577                 AS    Alaska Airlines
3165                 B6            JetBlue
3863                 DL              Delta
5694                 F9           Frontier
6106                 G4          Allegiant
6326                 HA           Hawaiian
7100                 NK             Spirit
9430                 UA             United
11149                WN          Southwest


In [31]:
print(flights.groupby('AIRLINE_NAME')[['FLYING_OPS', 'OP_EXPENSES']].mean().round(2))

                   FLYING_OPS  OP_EXPENSES
AIRLINE_NAME                              
Alaska Airlines     792515.37   2385138.45
Allegiant           271482.29    579461.03
American Airlines  2718718.47   8700286.53
Delta              2748631.93   9841441.41
Endeavor Air        100957.35    221801.14
Envoy Air           162694.12    461660.28
Frontier            531226.72    868757.98
Hawaiian            254898.75    631198.48
JetBlue             824382.25   1788448.41
PSA Airlines        143917.87    311093.62
Republic Airways    117018.77    329657.55
SkyWest             277284.99    683154.19
Southwest          2664174.77   6492875.32
Spirit              658479.91   1258294.89
United             2392491.64   7799930.66


In [32]:
flights['is_profitable']    = (flights['OP_PROFIT_LOSS'] > 0).astype('int8')

In [33]:
print("Delay rate by is_profitable:")
print(flights.groupby('is_profitable')['ARR_DEL15'].mean() * 100)

print("\nBy airline:")
print(flights.groupby('AIRLINE_NAME')[['is_profitable', 'ARR_DEL15']].mean().round(4).sort_values('ARR_DEL15', ascending=False))

Delay rate by is_profitable:
is_profitable
0    21.019581
1    21.346265
Name: ARR_DEL15, dtype: float64

By airline:
                   is_profitable  ARR_DEL15
AIRLINE_NAME                               
Frontier                  0.4918     0.2991
JetBlue                   0.2882     0.2789
Spirit                    0.0990     0.2544
American Airlines         1.0000     0.2514
Allegiant                 0.8400     0.2427
PSA Airlines              0.1858     0.2214
Alaska Airlines           0.7513     0.2173
Southwest                 0.6340     0.2128
United                    0.9138     0.2076
Envoy Air                 0.5517     0.2030
Hawaiian                  0.2876     0.1886
SkyWest                   0.6576     0.1875
Delta                     1.0000     0.1787
Republic Airways          1.0000     0.1545
Endeavor Air              0.0000     0.1487


In [34]:
print(flights.groupby('AIRLINE_NAME')[['profit_margin', 'maintenance_ratio', 
                                        'revenue_per_pax', 'cost_efficiency',
                                        'ARR_DEL15']].mean().round(4).sort_values('ARR_DEL15', ascending=False))

                   profit_margin  maintenance_ratio  revenue_per_pax  \
AIRLINE_NAME                                                           
Frontier                 -0.0068             0.0836           0.6233   
JetBlue                  -0.0992             0.1051           0.8561   
Spirit                   -0.1708             0.0607           0.4967   
American Airlines         0.0668             0.1124           0.6878   
Allegiant                 0.0695             0.1034           0.7015   
PSA Airlines             -0.0880             0.3136           0.9999   
Alaska Airlines           0.0528             0.0768           0.6997   
Southwest                 0.0063             0.0918           0.8629   
United                    0.0792             0.0935           0.7056   
Envoy Air                -0.0002             0.2039           0.9994   
Hawaiian                 -0.0617             0.1309           0.8676   
SkyWest                   0.0229             0.3367           0.

In [35]:
print("origin_sky_condition value distribution:")
print(flights['origin_sky_condition'].describe())
print("\nSample non-null values:")
print(flights['origin_sky_condition'].dropna().head(20).tolist())

origin_sky_condition value distribution:
count    9.717968e+06
mean     3.022750e+00
std      2.658904e+00
min      0.000000e+00
25%      0.000000e+00
50%      2.000000e+00
75%      4.000000e+00
max      9.000000e+00
Name: origin_sky_condition, dtype: float64

Sample non-null values:
[6.0, 4.0, 9.0, 8.0, 2.0, 6.0, 8.0, 9.0, 2.0, 8.0, 2.0, 9.0, 6.0, 8.0, 4.0, 4.0, 9.0, 4.0, 8.0, 8.0]


In [36]:
print("Delay rate by origin_sky_condition:")
print(flights.groupby('origin_sky_condition')['ARR_DEL15'].mean() * 100)

print("\nFlight counts per sky condition:")
print(flights['origin_sky_condition'].value_counts().sort_index())

Delay rate by origin_sky_condition:
origin_sky_condition
0.0    14.942751
1.0    20.714286
2.0    18.799977
3.0    11.538462
4.0    21.697569
6.0    23.361717
8.0    23.917772
9.0    28.719603
Name: ARR_DEL15, dtype: float64

Flight counts per sky condition:
origin_sky_condition
0.0    2726593
1.0        140
2.0    2760764
3.0         26
4.0    1915869
6.0    1203867
8.0    1029820
9.0      80889
Name: count, dtype: int64


In [ ]:
print("Null rate by year:")
print(flights.groupby('YEAR')['origin_sky_condition'].apply(lambda x: x.isna().mean() * 100).round(2))

print("\nNull rate for top 10 airports:")
print(flights.groupby('ORIGIN')['origin_sky_condition'].apply(lambda x: x.isna().mean() * 100).sort_values(ascending=False).head(10))

Null rate by year:
YEAR
2023    43.68
2024    41.80
2025    58.69
Name: origin_sky_condition, dtype: float64

Null rate for top 10 airports:
ORIGIN
ABE    100.0
MHK    100.0
MGM    100.0
MFR    100.0
MFE    100.0
MEI    100.0
MDT    100.0
MCW    100.0
MBS    100.0
MAF    100.0
Name: origin_sky_condition, dtype: float64


In [38]:
print("All unique sky condition values including dest:")
print(sorted(flights['dest_sky_condition'].dropna().unique().tolist()))

print("\nAirports reporting sky_condition=1:")
print(flights[flights['origin_sky_condition']==1]['ORIGIN'].value_counts().head(10))

print("\nAirports reporting sky_condition=3:")
print(flights[flights['origin_sky_condition']==3]['ORIGIN'].value_counts().head(10))

All unique sky condition values including dest:
[0.0, 1.0, 2.0, 3.0, 4.0, 6.0, 8.0, 9.0]

Airports reporting sky_condition=1:
ORIGIN
ORD    86
AUS    14
MCI    10
GSP     7
SAV     6
BOI     5
AVL     3
PVD     3
JAX     2
BDL     2
Name: count, dtype: int64

Airports reporting sky_condition=3:
ORIGIN
PVD    19
GSP     3
LGB     3
MSN     1
Name: count, dtype: int64


In [39]:
print(flights.FL_DATE.min())
print(flights.FL_DATE.max())

2023-01-01 00:00:00
2025-08-27 00:00:00


In [ ]:
flights['origin_sky_condition'] = flights['origin_sky_condition'].replace({1.0: 0.0, 3.0: 2.0})
flights['dest_sky_condition']   = flights['dest_sky_condition'].replace({1.0: 0.0, 3.0: 2.0})

print("origin_sky_condition values after merge:")
print(flights['origin_sky_condition'].value_counts().sort_index())

print("\nDelay rate after merge:")
print(flights.groupby('origin_sky_condition')['ARR_DEL15'].mean() * 100)

origin_sky_condition values after merge:
origin_sky_condition
0.0    2726733
2.0    2760790
4.0    1915869
6.0    1203867
8.0    1029820
9.0      80889
Name: count, dtype: int64

Delay rate after merge:
origin_sky_condition
0.0    14.943047
2.0    18.799909
4.0    21.697569
6.0    23.361717
8.0    23.917772
9.0    28.719603
Name: ARR_DEL15, dtype: float64


In [41]:
flights['origin_is_adverse_sky'] = flights['origin_sky_condition'].apply(
    lambda x: 1 if x >= 6 else (0 if pd.notna(x) else np.nan)
).astype('float32')

flights['dest_is_adverse_sky'] = flights['dest_sky_condition'].apply(
    lambda x: 1 if x >= 6 else (0 if pd.notna(x) else np.nan)
).astype('float32')

print("Delay rate by origin_is_adverse_sky:")
print(flights.groupby('origin_is_adverse_sky')['ARR_DEL15'].mean() * 100)

print("\nValue counts:")
print(flights['origin_is_adverse_sky'].value_counts(dropna=False))

Delay rate by origin_is_adverse_sky:
origin_is_adverse_sky
0.0    18.129258
1.0    23.796367
Name: ARR_DEL15, dtype: float64

Value counts:
origin_is_adverse_sky
NaN    8509828
0.0    7403392
1.0    2314576
Name: count, dtype: int64


In [42]:
print("origin_sky_condition:")
print(f"  dtype: {flights['origin_sky_condition'].dtype}")
print(f"  unique values: {sorted(flights['origin_sky_condition'].dropna().unique().tolist())}")
print(f"  nulls: {flights['origin_sky_condition'].isna().sum():,}")

print("\norigin_is_adverse_sky:")
print(f"  dtype: {flights['origin_is_adverse_sky'].dtype}")
print(f"  unique values: {flights['origin_is_adverse_sky'].value_counts(dropna=False).to_dict()}")
print(f"  nulls: {flights['origin_is_adverse_sky'].isna().sum():,}")

print("\ndest_sky_condition:")
print(f"  dtype: {flights['dest_sky_condition'].dtype}")
print(f"  unique values: {sorted(flights['dest_sky_condition'].dropna().unique().tolist())}")
print(f"  nulls: {flights['dest_sky_condition'].isna().sum():,}")

print("\ndest_is_adverse_sky:")
print(f"  dtype: {flights['dest_is_adverse_sky'].dtype}")
print(f"  unique values: {flights['dest_is_adverse_sky'].value_counts(dropna=False).to_dict()}")
print(f"  nulls: {flights['dest_is_adverse_sky'].isna().sum():,}")

print(f"\nShape: {flights.shape}")

origin_sky_condition:
  dtype: float64
  unique values: [0.0, 2.0, 4.0, 6.0, 8.0, 9.0]
  nulls: 8,509,828

origin_is_adverse_sky:
  dtype: float32
  unique values: {nan: 8509828, 0.0: 7403392, 1.0: 2314576}
  nulls: 8,509,828

dest_sky_condition:
  dtype: float64
  unique values: [0.0, 2.0, 4.0, 6.0, 8.0, 9.0]
  nulls: 8,738,299

dest_is_adverse_sky:
  dtype: float32
  unique values: {nan: 8738299, 0.0: 7247404, 1.0: 2242093}
  nulls: 8,738,299

Shape: (18227796, 86)


In [43]:
print("Delay rate by year:")
print(flights.groupby('YEAR')['ARR_DEL15'].mean() * 100)

print("\nFlight counts by year:")
print(flights.groupby('YEAR')['ARR_DEL15'].count())

Delay rate by year:
YEAR
2023    20.563787
2024    20.817178
2025    22.965414
Name: ARR_DEL15, dtype: float64

Flight counts by year:
YEAR
2023    6743403
2024    6965267
2025    4519126
Name: ARR_DEL15, dtype: int64


In [44]:
print("TAIL_NUM sample:")
print(flights['TAIL_NUM'].value_counts().head(10))

print("\nNull tail numbers:")
print(flights['TAIL_NUM'].isna().sum())

print("\nHow many flights per tail per day on average:")
tail_daily = flights.groupby(['TAIL_NUM', 'FL_DATE']).size()
print(tail_daily.describe())

TAIL_NUM sample:
TAIL_NUM
N488HA    8277
N495HA    8145
N492HA    8082
N489HA    8005
N486HA    7948
N477HA    7915
N483HA    7908
N485HA    7894
N490HA    7790
N493HA    7781
Name: count, dtype: int64

Null tail numbers:
0

How many flights per tail per day on average:
count    4.516145e+06
mean     4.036141e+00
std      1.728746e+00
min      1.000000e+00
25%      3.000000e+00
50%      4.000000e+00
75%      5.000000e+00
max      1.400000e+01
dtype: float64


In [ ]:
sample_tail = flights.groupby(['TAIL_NUM', 'FL_DATE']).size().idxmax()
print(f"Busiest tail+day: {sample_tail}")

sample = flights[
    (flights['TAIL_NUM'] == sample_tail[0]) & 
    (flights['FL_DATE'] == sample_tail[1])
][['TAIL_NUM', 'FL_DATE', 'CRS_DEP_TIME', 'ORIGIN', 'DEST', 
   'ARR_DEL15', 'ARR_DELAY', 'LATE_AIRCRAFT_DELAY']].sort_values('CRS_DEP_TIME')

print(sample.to_string())

Busiest tail+day: ('N476HA', Timestamp('2024-01-12 00:00:00'))
         TAIL_NUM    FL_DATE  CRS_DEP_TIME ORIGIN DEST  ARR_DEL15  ARR_DELAY  LATE_AIRCRAFT_DELAY
13383190   N476HA 2024-01-12           536    HNL  LIH        0.0        4.0                  NaN
13383192   N476HA 2024-01-12           645    LIH  HNL        0.0       -1.0                  NaN
13383194   N476HA 2024-01-12           800    HNL  KOA        0.0       -3.0                  NaN
13383193   N476HA 2024-01-12           917    KOA  LIH        0.0        1.0                  NaN
13383195   N476HA 2024-01-12          1041    LIH  HNL        0.0        2.0                  NaN
13383197   N476HA 2024-01-12          1209    HNL  LIH        0.0        3.0                  NaN
13383196   N476HA 2024-01-12          1320    LIH  HNL        0.0        0.0                  NaN
13383199   N476HA 2024-01-12          1440    HNL  LIH        1.0       15.0                  0.0
13383198   N476HA 2024-01-12          1550    LIH  HNL 

In [ ]:
sample_tail2 = 'N476HA'

overnight = flights[
    flights['TAIL_NUM'] == sample_tail2
][['TAIL_NUM', 'FL_DATE', 'CRS_DEP_TIME', 'ORIGIN', 'DEST',
   'ARR_DEL15', 'ARR_DELAY', 'LATE_AIRCRAFT_DELAY']].sort_values(['FL_DATE', 'CRS_DEP_TIME'])
   
jan12_last = overnight[overnight['FL_DATE'] == '2024-01-12'].tail(1)
jan13_first = overnight[overnight['FL_DATE'] == '2024-01-13'].head(1)

print("Last flight Jan 12:")
print(jan12_last.to_string())
print("\nFirst flight Jan 13:")
print(jan13_first.to_string())

Last flight Jan 12:
         TAIL_NUM    FL_DATE  CRS_DEP_TIME ORIGIN DEST  ARR_DEL15  ARR_DELAY  LATE_AIRCRAFT_DELAY
13383191   N476HA 2024-01-12          2200    OGG  HNL        1.0      129.0                114.0

First flight Jan 13:
         TAIL_NUM    FL_DATE  CRS_DEP_TIME ORIGIN DEST  ARR_DEL15  ARR_DELAY  LATE_AIRCRAFT_DELAY
13398560   N476HA 2024-01-13           700    HNL  LIH        0.0       -2.0                  NaN


In [ ]:
flights_sorted = flights[['TAIL_NUM', 'FL_DATE', 'CRS_DEP_TIME', 
                           'ARR_DEL15', 'ARR_DELAY', 'LATE_AIRCRAFT_DELAY']].copy()

flights_sorted['DEP_DATETIME'] = pd.to_datetime(
    flights_sorted['FL_DATE'].astype(str)
) + pd.to_timedelta(
    flights_sorted['CRS_DEP_TIME'].astype(str).str.zfill(4).str[:2].astype(int) * 60 + 
    flights_sorted['CRS_DEP_TIME'].astype(str).str.zfill(4).str[2:].astype(int), 
    unit='m'
)

flights_sorted = flights_sorted.sort_values(['TAIL_NUM', 'DEP_DATETIME'])

flights_sorted['prev_dep_datetime'] = flights_sorted.groupby('TAIL_NUM')['DEP_DATETIME'].shift(1)
flights_sorted['prev_delayed'] = flights_sorted.groupby('TAIL_NUM')['ARR_DEL15'].shift(1)
flights_sorted['gap_hours'] = (
    flights_sorted['DEP_DATETIME'] - flights_sorted['prev_dep_datetime']
).dt.total_seconds() / 3600

delayed_prev = flights_sorted[
    (flights_sorted['prev_delayed'] == 1.0) & 
    (flights_sorted['gap_hours'].between(0, 24))
].copy()

print("When previous flight was delayed, delay rate of next flight by gap size:")
delayed_prev['gap_bucket'] = pd.cut(delayed_prev['gap_hours'], 
                                     bins=[0,1,2,3,4,5,6,8,10,12,24])
print(delayed_prev.groupby('gap_bucket')['ARR_DEL15'].mean() * 100)
print("\nFlight counts per gap bucket:")
print(delayed_prev.groupby('gap_bucket')['ARR_DEL15'].count())

When previous flight was delayed, delay rate of next flight by gap size:
gap_bucket
(0, 1]      54.653927
(1, 2]      74.759454
(2, 3]      69.207358
(3, 4]      65.384085
(4, 5]      60.409729
(5, 6]      53.407053
(6, 8]      36.269962
(8, 10]     22.271657
(10, 12]    15.670886
(12, 24]    21.134993
Name: ARR_DEL15, dtype: float64

Flight counts per gap bucket:
gap_bucket
(0, 1]       43329
(1, 2]      356585
(2, 3]      837541
(3, 4]      595154
(4, 5]      349548
(5, 6]      140532
(6, 8]      232625
(8, 10]     323658
(10, 12]    330779
(12, 24]    537360
Name: ARR_DEL15, dtype: int64


/var/folders/_r/scvy88yj1wl9m4x325v7q1200000gn/T/ipykernel_8232/3310160766.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(delayed_prev.groupby('gap_bucket')['ARR_DEL15'].mean() * 100)
/var/folders/_r/scvy88yj1wl9m4x325v7q1200000gn/T/ipykernel_8232/3310160766.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(delayed_prev.groupby('gap_bucket')['ARR_DEL15'].count())


In [ ]:
check = flights_sorted[flights_sorted['TAIL_NUM'] == 'N476HA'].sort_values('DEP_DATETIME')
print(check[['FL_DATE', 'CRS_DEP_TIME', 'DEP_DATETIME', 'ARR_DEL15', 'ARR_DELAY']].head(15).to_string())

         FL_DATE  CRS_DEP_TIME        DEP_DATETIME  ARR_DEL15  ARR_DELAY
6401  2023-01-01           630 2023-01-01 06:30:00        0.0       -8.0
6402  2023-01-01           755 2023-01-01 07:55:00        0.0      -15.0
6404  2023-01-01           930 2023-01-01 09:30:00        0.0       -8.0
6403  2023-01-01          1110 2023-01-01 11:10:00        0.0      -12.0
6405  2023-01-01          1320 2023-01-01 13:20:00        0.0        5.0
6407  2023-01-01          1440 2023-01-01 14:40:00        0.0        6.0
6406  2023-01-01          1550 2023-01-01 15:50:00        0.0       11.0
6408  2023-01-01          1730 2023-01-01 17:30:00        0.0       -7.0
6409  2023-01-01          1845 2023-01-01 18:45:00        0.0       -6.0
6410  2023-01-01          2015 2023-01-01 20:15:00        0.0       -7.0
23183 2023-01-02           915 2023-01-02 09:15:00        0.0       -2.0
23184 2023-01-02          1100 2023-01-02 11:00:00        0.0      -11.0
23186 2023-01-02          1250 2023-01-02 12:50:00 

In [49]:
print("DEP_DATETIME in flights?", 'DEP_DATETIME' in flights.columns)
print("DEP_DATETIME in flights_sorted?", 'DEP_DATETIME' in flights_sorted.columns)

DEP_DATETIME in flights? False
DEP_DATETIME in flights_sorted? True


In [ ]:
flights['DEP_DATETIME'] = pd.to_datetime(
    flights['FL_DATE'].astype(str)
) + pd.to_timedelta(
    flights['CRS_DEP_TIME'].astype(str).str.zfill(4).str[:2].astype(int) * 60 + 
    flights['CRS_DEP_TIME'].astype(str).str.zfill(4).str[2:].astype(int), 
    unit='m'
)

check = flights[flights['TAIL_NUM'] == 'N476HA'].sort_values('DEP_DATETIME')
print(check[['FL_DATE', 'CRS_DEP_TIME', 'DEP_DATETIME']].head(5).to_string())
print(f"\nDEP_DATETIME dtype: {flights['DEP_DATETIME'].dtype}")
print(f"Nulls: {flights['DEP_DATETIME'].isna().sum()}")
print(f"Shape: {flights.shape}")

        FL_DATE  CRS_DEP_TIME        DEP_DATETIME
6401 2023-01-01           630 2023-01-01 06:30:00
6402 2023-01-01           755 2023-01-01 07:55:00
6404 2023-01-01           930 2023-01-01 09:30:00
6403 2023-01-01          1110 2023-01-01 11:10:00
6405 2023-01-01          1320 2023-01-01 13:20:00

DEP_DATETIME dtype: datetime64[ns]
Nulls: 0
Shape: (18227796, 87)


In [ ]:
hawaiian_tails = flights[flights['OP_UNIQUE_CARRIER'] == 'HA']['TAIL_NUM'].unique()
print(f"Hawaiian tails: {len(hawaiian_tails)}")

sample = flights[flights['TAIL_NUM'].isin(hawaiian_tails)][
    ['TAIL_NUM', 'FL_DATE', 'CRS_DEP_TIME', 'DEP_DATETIME', 
     'ORIGIN', 'DEST', 'ARR_DEL15', 'ARR_DELAY']
].copy()

sample = sample.sort_values(['TAIL_NUM', 'DEP_DATETIME'])
print(f"Sample shape: {sample.shape}")
print(f"Flights per tail per day:")
print(sample.groupby(['TAIL_NUM', 'FL_DATE']).size().describe())

Hawaiian tails: 65
Sample shape: (210025, 8)
Flights per tail per day:
count    50924.000000
mean         4.124283
std          3.618071
min          1.000000
25%          2.000000
50%          2.000000
75%          8.000000
max         14.000000
dtype: float64


In [ ]:
test = sample[sample['TAIL_NUM'] == 'N476HA'].copy()
test = test.sort_values('DEP_DATETIME').reset_index(drop=True)

print(f"N476HA total flights: {len(test)}")
print(f"\nJan 12 2024 flights:")
print(test[test['FL_DATE'] == '2024-01-12'][
    ['FL_DATE', 'CRS_DEP_TIME', 'ORIGIN', 'DEST', 
     'ARR_DEL15', 'ARR_DELAY']
].to_string())

N476HA total flights: 7612

Jan 12 2024 flights:
        FL_DATE  CRS_DEP_TIME ORIGIN DEST  ARR_DEL15  ARR_DELAY
2781 2024-01-12           536    HNL  LIH        0.0        4.0
2782 2024-01-12           645    LIH  HNL        0.0       -1.0
2783 2024-01-12           800    HNL  KOA        0.0       -3.0
2784 2024-01-12           917    KOA  LIH        0.0        1.0
2785 2024-01-12          1041    LIH  HNL        0.0        2.0
2786 2024-01-12          1209    HNL  LIH        0.0        3.0
2787 2024-01-12          1320    LIH  HNL        0.0        0.0
2788 2024-01-12          1440    HNL  LIH        1.0       15.0
2789 2024-01-12          1550    LIH  HNL        1.0       20.0
2790 2024-01-12          1705    HNL  OGG        1.0       28.0
2791 2024-01-12          1815    OGG  HNL        1.0       36.0
2792 2024-01-12          1930    HNL  LIH        1.0       37.0
2793 2024-01-12          2039    LIH  HNL        1.0       25.0
2794 2024-01-12          2200    OGG  HNL        1.0   

In [ ]:
test = sample[sample['TAIL_NUM'] == 'N476HA'].copy()
test = test.sort_values('DEP_DATETIME').reset_index(drop=True)

test['prev_dep']      = test['DEP_DATETIME'].shift(1)
test['prev_delayed']  = test['ARR_DEL15'].shift(1)
test['prev_delay_min']= test['ARR_DELAY'].shift(1)

test['gap_hrs'] = (
    test['DEP_DATETIME'] - test['prev_dep']
).dt.total_seconds() / 3600

test['cascade_score'] = (
    (test['prev_delayed'] == 1.0) & 
    (test['gap_hrs'] <= 6) & 
    (test['gap_hrs'] > 0)
).astype('int8')

test['cascade_delay_minutes'] = np.where(
    (test['gap_hrs'] <= 6) & (test['gap_hrs'] > 0),
    test['prev_delay_min'].clip(lower=0).fillna(0),
    0
).astype('float32')

test['hours_since_last_delay'] = np.where(
    (test['prev_delayed'] == 1.0) & (test['gap_hrs'] <= 6),
    test['gap_hrs'],
    np.nan
).astype('float32')

print("Jan 12 2024 cascade scores:")
print(test[test['FL_DATE'] == '2024-01-12'][
    ['CRS_DEP_TIME', 'ORIGIN', 'DEST', 'ARR_DEL15', 'ARR_DELAY',
     'gap_hrs', 'cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay']
].to_string())

Jan 12 2024 cascade scores:
      CRS_DEP_TIME ORIGIN DEST  ARR_DEL15  ARR_DELAY   gap_hrs  cascade_score  cascade_delay_minutes  hours_since_last_delay
2781           536    HNL  LIH        0.0        4.0  8.950000              0                    0.0                     NaN
2782           645    LIH  HNL        0.0       -1.0  1.150000              0                    4.0                     NaN
2783           800    HNL  KOA        0.0       -3.0  1.250000              0                    0.0                     NaN
2784           917    KOA  LIH        0.0        1.0  1.283333              0                    0.0                     NaN
2785          1041    LIH  HNL        0.0        2.0  1.400000              0                    1.0                     NaN
2786          1209    HNL  LIH        0.0        3.0  1.466667              0                    2.0                     NaN
2787          1320    LIH  HNL        0.0        0.0  1.183333              0                    

In [54]:
test['cascade_delay_minutes'] = np.where(
    (test['prev_delayed'] == 1.0) & (test['gap_hrs'] <= 6) & (test['gap_hrs'] > 0),
    test['prev_delay_min'].clip(lower=0).fillna(0),
    0
).astype('float32')

print("Jan 12 2024 after fix:")
print(test[test['FL_DATE'] == '2024-01-12'][
    ['CRS_DEP_TIME', 'ARR_DEL15', 'ARR_DELAY',
     'cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay']
].to_string())

Jan 12 2024 after fix:
      CRS_DEP_TIME  ARR_DEL15  ARR_DELAY  cascade_score  cascade_delay_minutes  hours_since_last_delay
2781           536        0.0        4.0              0                    0.0                     NaN
2782           645        0.0       -1.0              0                    0.0                     NaN
2783           800        0.0       -3.0              0                    0.0                     NaN
2784           917        0.0        1.0              0                    0.0                     NaN
2785          1041        0.0        2.0              0                    0.0                     NaN
2786          1209        0.0        3.0              0                    0.0                     NaN
2787          1320        0.0        0.0              0                    0.0                     NaN
2788          1440        1.0       15.0              0                    0.0                     NaN
2789          1550        1.0       20.0          

In [ ]:
def compute_cascade(tail_df):
    """
    For each flight in a tail's history,
    look back 6 hours and compute cascade features.
    """
    tail_df = tail_df.sort_values('DEP_DATETIME').copy()
    
    n = len(tail_df)
    cascade_score         = np.zeros(n, dtype='int8')
    cascade_delay_minutes = np.zeros(n, dtype='float32')
    hours_since_last      = np.full(n, np.nan, dtype='float32')
    
    dep_times  = tail_df['DEP_DATETIME'].values
    delayed    = tail_df['ARR_DEL15'].values
    delay_mins = tail_df['ARR_DELAY'].values
    
    for i in range(1, n):
        window_start = dep_times[i] - np.timedelta64(6, 'h')
        
        for j in range(i-1, -1, -1):
            if dep_times[j] < window_start:
                break  
            if delayed[j] == 1.0:
                cascade_score[i] += 1
                cascade_delay_minutes[i] += max(0, delay_mins[j])
                if np.isnan(hours_since_last[i]):

                    hours_since_last[i] = (
                        dep_times[i] - dep_times[j]
                    ) / np.timedelta64(1, 'h')
    
    tail_df['cascade_score']          = cascade_score
    tail_df['cascade_delay_minutes']  = cascade_delay_minutes
    tail_df['hours_since_last_delay'] = hours_since_last
    return tail_df

test = sample[sample['TAIL_NUM'] == 'N476HA'].copy()
result = compute_cascade(test)

print("Jan 12 2024 — full 6hr window cascade:")
print(result[result['FL_DATE'] == '2024-01-12'][
    ['CRS_DEP_TIME', 'ARR_DEL15', 'ARR_DELAY',
     'cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay']
].to_string())

Jan 12 2024 — full 6hr window cascade:
          CRS_DEP_TIME  ARR_DEL15  ARR_DELAY  cascade_score  cascade_delay_minutes  hours_since_last_delay
13383190           536        0.0        4.0              0                    0.0                     NaN
13383192           645        0.0       -1.0              0                    0.0                     NaN
13383194           800        0.0       -3.0              0                    0.0                     NaN
13383193           917        0.0        1.0              0                    0.0                     NaN
13383195          1041        0.0        2.0              0                    0.0                     NaN
13383197          1209        0.0        3.0              0                    0.0                     NaN
13383196          1320        0.0        0.0              0                    0.0                     NaN
13383199          1440        1.0       15.0              0                    0.0                     Na

In [56]:
import time

start = time.time()
hawaiian_result = sample.groupby('TAIL_NUM', group_keys=False).apply(compute_cascade)
elapsed = time.time() - start

print(f"Hawaiian tails (210K flights): {elapsed:.1f} seconds")
print(f"Shape: {hawaiian_result.shape}")
print(f"\ncascade_score distribution:")
print(hawaiian_result['cascade_score'].value_counts().sort_index())

Hawaiian tails (210K flights): 0.9 seconds
Shape: (210025, 11)

cascade_score distribution:
cascade_score
0    175492
1     18342
2      8085
3      4980
4      3097
5        29
Name: count, dtype: int64


/var/folders/_r/scvy88yj1wl9m4x325v7q1200000gn/T/ipykernel_8232/3541960923.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  hawaiian_result = sample.groupby('TAIL_NUM', group_keys=False).apply(compute_cascade)


In [57]:
import time

start = time.time()
hawaiian_result = sample.groupby('TAIL_NUM', group_keys=False).apply(
    compute_cascade, include_groups=False
)
elapsed = time.time() - start

print(f"Time: {elapsed:.1f} seconds")
print(f"Shape: {hawaiian_result.shape}")

Time: 0.7 seconds
Shape: (210025, 10)


In [ ]:
def compute_cascade(tail_df):
    tail_df = tail_df.sort_values('DEP_DATETIME').copy()
    
    n = len(tail_df)
    cascade_score         = np.zeros(n, dtype='int8')
    cascade_delay_minutes = np.zeros(n, dtype='float32')
    hours_since_last      = np.full(n, np.nan, dtype='float32')
    
    dep_times  = tail_df['DEP_DATETIME'].values
    delayed    = tail_df['ARR_DEL15'].values
    delay_mins = tail_df['ARR_DELAY'].values
    
    for i in range(1, n):
        window_start = dep_times[i] - np.timedelta64(6, 'h')
        for j in range(i-1, -1, -1):
            if dep_times[j] < window_start:
                break
            if delayed[j] == 1.0:
                cascade_score[i] += 1
                cascade_delay_minutes[i] += max(0, delay_mins[j])
                if np.isnan(hours_since_last[i]):
                    hours_since_last[i] = (
                        dep_times[i] - dep_times[j]
                    ) / np.timedelta64(1, 'h')
    
    tail_df['cascade_score']          = cascade_score
    tail_df['cascade_delay_minutes']  = cascade_delay_minutes
    tail_df['hours_since_last_delay'] = hours_since_last
    return tail_df

print("Running cascade on full 18M rows...")
start = time.time()

cascade_results = flights[['TAIL_NUM', 'DEP_DATETIME', 'ARR_DEL15', 'ARR_DELAY']].copy()
cascade_results = cascade_results.groupby('TAIL_NUM', group_keys=False).apply(compute_cascade)

elapsed = time.time() - start
print(f"Done in {elapsed:.1f} seconds")
print(f"\ncascade_score distribution:")
print(cascade_results['cascade_score'].value_counts().sort_index())
print(f"\nDelay rate by cascade_score:")
print(cascade_results.groupby('cascade_score')['ARR_DEL15'].mean() * 100)

Running cascade on full 18M rows...
Done in 48.0 seconds

cascade_score distribution:
cascade_score
0    15649459
1     2053748
2      481206
3       39437
4        3880
5          65
6           1
Name: count, dtype: int64

Delay rate by cascade_score:
cascade_score
0     14.503358
1     58.966923
2     74.437975
3     81.702462
4     84.871134
5     96.923077
6    100.000000
Name: ARR_DEL15, dtype: float64


/var/folders/_r/scvy88yj1wl9m4x325v7q1200000gn/T/ipykernel_8232/3849574310.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cascade_results = cascade_results.groupby('TAIL_NUM', group_keys=False).apply(compute_cascade)


In [ ]:
flights['cascade_score'] = cascade_results['cascade_score'].values
flights['cascade_delay_minutes'] = cascade_results['cascade_delay_minutes'].values
flights['hours_since_last_delay'] = cascade_results['hours_since_last_delay'].values

print(f"Shape: {flights.shape}")
print(f"\nVerify on N476HA Jan 12:")
print(flights[
    (flights['TAIL_NUM'] == 'N476HA') & 
    (flights['FL_DATE'] == '2024-01-12')
][['CRS_DEP_TIME', 'ARR_DEL15', 'ARR_DELAY',
   'cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay']].to_string())

Shape: (18227796, 90)

Verify on N476HA Jan 12:
          CRS_DEP_TIME  ARR_DEL15  ARR_DELAY  cascade_score  cascade_delay_minutes  hours_since_last_delay
13383190           536        0.0        4.0              0                    0.0                     NaN
13383191          2200        1.0      129.0              0                    0.0                     NaN
13383192           645        0.0       -1.0              0                    0.0                     NaN
13383193           917        0.0        1.0              0                    0.0                     NaN
13383194           800        0.0       -3.0              0                    0.0                     NaN
13383195          1041        0.0        2.0              0                    0.0                     NaN
13383196          1320        0.0        0.0              0                    0.0                     NaN
13383197          1209        0.0        3.0              0                    0.0              

In [60]:
print("flights index sample:", flights.index[:5].tolist())
print("cascade_results index sample:", cascade_results.index[:5].tolist())
print("\nAre indices aligned?", (flights.index == cascade_results.index).all())

flights index sample: [0, 1, 2, 3, 4]
cascade_results index sample: [13646468, 8516041, 8516042, 8534322, 8534320]

Are indices aligned? False


In [ ]:
flights.drop(columns=['cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay'], 
             inplace=True)

cascade_to_join = cascade_results[['cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay']]

flights = flights.join(cascade_to_join, how='left')

print(f"Shape: {flights.shape}")
print(f"\nVerify on N476HA Jan 12:")
print(flights[
    (flights['TAIL_NUM'] == 'N476HA') & 
    (flights['FL_DATE'] == '2024-01-12')
][['CRS_DEP_TIME', 'ARR_DEL15', 'ARR_DELAY',
   'cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay']
].sort_values('CRS_DEP_TIME').to_string())

Shape: (18227796, 90)

Verify on N476HA Jan 12:
          CRS_DEP_TIME  ARR_DEL15  ARR_DELAY  cascade_score  cascade_delay_minutes  hours_since_last_delay
13383190           536        0.0        4.0              0                    0.0                     NaN
13383192           645        0.0       -1.0              0                    0.0                     NaN
13383194           800        0.0       -3.0              0                    0.0                     NaN
13383193           917        0.0        1.0              0                    0.0                     NaN
13383195          1041        0.0        2.0              0                    0.0                     NaN
13383197          1209        0.0        3.0              0                    0.0                     NaN
13383196          1320        0.0        0.0              0                    0.0                     NaN
13383199          1440        1.0       15.0              0                    0.0              

In [62]:
flights.to_parquet("dataset/merged_flights_fe.parquet", 
                   index=False, engine="pyarrow", compression="snappy")
import os
size_mb = os.path.getsize("dataset/merged_flights_fe.parquet") / 1024 / 1024
print(f"✅ Saved. Size: {size_mb:.1f} MB | Shape: {flights.shape}")

✅ Saved. Size: 839.1 MB | Shape: (18227796, 90)


In [ ]:
interesting = flights[flights['cascade_score'] >= 3][['TAIL_NUM', 'FL_DATE']].value_counts().head(5)
print("Tails with highest cascade scores:")
print(interesting)

Tails with highest cascade scores:
TAIL_NUM  FL_DATE   
N488HA    2025-01-31    9
N485HA    2023-05-06    9
N488HA    2023-05-13    8
N475HA    2025-01-31    8
N486HA    2024-02-16    8
Name: count, dtype: int64


In [64]:
verify = flights[
    (flights['TAIL_NUM'] == 'N488HA') & 
    (flights['FL_DATE'] == '2025-01-31')
][['CRS_DEP_TIME', 'ORIGIN', 'DEST', 'ARR_DEL15', 'ARR_DELAY',
   'cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay']
].sort_values('CRS_DEP_TIME')

print(verify.to_string())

          CRS_DEP_TIME ORIGIN DEST  ARR_DEL15  ARR_DELAY  cascade_score  cascade_delay_minutes  hours_since_last_delay
18217508           536    HNL  OGG        0.0        8.0              0                    0.0                     NaN
18217507           645    OGG  HNL        1.0       23.0              0                    0.0                     NaN
18217510           800    HNL  LIH        1.0       32.0              1                   23.0                1.250000
18217509           915    LIH  OGG        1.0       33.0              2                   55.0                1.250000
18217511          1036    OGG  HNL        1.0       51.0              3                   88.0                1.350000
18217513          1155    HNL  KOA        1.0       55.0              4                  139.0                1.316667
18217512          1315    KOA  HNL        1.0      117.0              4                  171.0                1.333333
18217515          1505    HNL  OGG        1.0   

In [65]:
print("Unique carrier+date combinations:")
print(flights.groupby(['OP_UNIQUE_CARRIER', 'FL_DATE']).ngroups)

print("\nDate range:")
print(f"Min: {flights['FL_DATE'].min()}")
print(f"Max: {flights['FL_DATE'].max()}")

Unique carrier+date combinations:
14311

Date range:
Min: 2023-01-01 00:00:00
Max: 2025-08-27 00:00:00


In [ ]:
carrier_daily = (
    flights.groupby(['OP_UNIQUE_CARRIER', 'FL_DATE'])['ARR_DEL15']
    .mean()
    .reset_index()
    .rename(columns={'ARR_DEL15': 'daily_delay_rate'})
)

print(f"carrier_daily shape: {carrier_daily.shape}")
print(carrier_daily.head(10).to_string())

carrier_daily shape: (14311, 3)
  OP_UNIQUE_CARRIER    FL_DATE  daily_delay_rate
0                9E 2023-01-01          0.126437
1                9E 2023-01-02          0.134703
2                9E 2023-01-03          0.181333
3                9E 2023-01-04          0.289786
4                9E 2023-01-05          0.132075
5                9E 2023-01-06          0.065728
6                9E 2023-01-07          0.072941
7                9E 2023-01-08          0.121528
8                9E 2023-01-09          0.104575
9                9E 2023-01-10          0.125203


In [ ]:
carrier_daily = carrier_daily.sort_values(['OP_UNIQUE_CARRIER', 'FL_DATE'])

carrier_daily['airline_delay_rate_30d'] = (
    carrier_daily
    .groupby('OP_UNIQUE_CARRIER')['daily_delay_rate']
    .transform(lambda x: x.shift(1).rolling(window=30, min_periods=7).mean())
)

print(f"Shape: {carrier_daily.shape}")
print(f"\nNulls in airline_delay_rate_30d: {carrier_daily['airline_delay_rate_30d'].isna().sum()}")
print(f"\nSample for American Airlines:")
print(carrier_daily[carrier_daily['OP_UNIQUE_CARRIER']=='AA'].head(35)[
    ['FL_DATE','daily_delay_rate','airline_delay_rate_30d']
].to_string())



Shape: (14311, 4)

Nulls in airline_delay_rate_30d: 105

Sample for American Airlines:
       FL_DATE  daily_delay_rate  airline_delay_rate_30d
731 2023-01-01          0.190220                     NaN
732 2023-01-02          0.355263                     NaN
733 2023-01-03          0.458251                     NaN
734 2023-01-04          0.382796                     NaN
735 2023-01-05          0.257155                     NaN
736 2023-01-06          0.167886                     NaN
737 2023-01-07          0.149509                     NaN
738 2023-01-08          0.138564                0.280154
739 2023-01-09          0.136238                0.262455
740 2023-01-10          0.136402                0.248431
741 2023-01-11          0.775510                0.237228
742 2023-01-12          0.323272                0.286163
743 2023-01-13          0.272799                0.289256
744 2023-01-14          0.124542                0.287990
745 2023-01-15          0.148702                0.276315
7

In [ ]:
flights = flights.merge(
    carrier_daily[['OP_UNIQUE_CARRIER', 'FL_DATE', 'airline_delay_rate_30d']],
    on=['OP_UNIQUE_CARRIER', 'FL_DATE'],
    how='left'
)

print(f"Shape: {flights.shape}")
print(f"Nulls in airline_delay_rate_30d: {flights['airline_delay_rate_30d'].isna().sum():,}")
print(f"\nDelay rate correlation:")
print(flights.groupby('OP_UNIQUE_CARRIER')[['airline_delay_rate_30d', 'ARR_DEL15']].mean().round(4))

Shape: (18227796, 91)
Nulls in airline_delay_rate_30d: 114,775

Delay rate correlation:
                   airline_delay_rate_30d  ARR_DEL15
OP_UNIQUE_CARRIER                                   
9E                                 0.1496     0.1487
AA                                 0.2509     0.2514
AS                                 0.2161     0.2173
B6                                 0.2796     0.2789
DL                                 0.1801     0.1787
F9                                 0.2954     0.2991
G4                                 0.2237     0.2427
HA                                 0.1905     0.1886
MQ                                 0.2047     0.2030
NK                                 0.2538     0.2544
OH                                 0.2226     0.2214
OO                                 0.1875     0.1875
UA                                 0.2085     0.2076
WN                                 0.2113     0.2128
YX                                 0.1573     0.1545


In [ ]:
origin_daily = (
    flights.groupby(['ORIGIN', 'FL_DATE'])['ARR_DEL15']
    .mean()
    .reset_index()
    .rename(columns={'ARR_DEL15': 'daily_delay_rate'})
)

origin_daily = origin_daily.sort_values(['ORIGIN', 'FL_DATE'])

origin_daily['origin_delay_rate_30d'] = (
    origin_daily
    .groupby('ORIGIN')['daily_delay_rate']
    .transform(lambda x: x.shift(1).rolling(window=30, min_periods=7).mean())
)

print(f"origin_daily shape: {origin_daily.shape}")
print(f"Nulls: {origin_daily['origin_delay_rate_30d'].isna().sum():,}")
print(f"\nSample for DFW:")
print(origin_daily[origin_daily['ORIGIN']=='DFW'].head(10).to_string())

origin_daily shape: (317059, 4)
Nulls: 2,521

Sample for DFW:
      ORIGIN    FL_DATE  daily_delay_rate  origin_delay_rate_30d
82938    DFW 2023-01-01          0.165217                    NaN
82939    DFW 2023-01-02          0.417027                    NaN
82940    DFW 2023-01-03          0.442424                    NaN
82941    DFW 2023-01-04          0.312977                    NaN
82942    DFW 2023-01-05          0.207752                    NaN
82943    DFW 2023-01-06          0.172573                    NaN
82944    DFW 2023-01-07          0.118902                    NaN
82945    DFW 2023-01-08          0.153957               0.262411
82946    DFW 2023-01-09          0.130307               0.248854
82947    DFW 2023-01-10          0.137008               0.235682


In [70]:
flights = flights.merge(
    origin_daily[['ORIGIN', 'FL_DATE', 'origin_delay_rate_30d']],
    on=['ORIGIN', 'FL_DATE'],
    how='left'
)

print(f"Shape: {flights.shape}")
print(f"Nulls in origin_delay_rate_30d: {flights['origin_delay_rate_30d'].isna().sum():,}")
print(f"\nTop 5 worst airports by rolling delay rate:")
print(flights.groupby('ORIGIN')[['origin_delay_rate_30d', 'ARR_DEL15']].mean().round(4).sort_values('ARR_DEL15', ascending=False).head())

Shape: (18227796, 92)
Nulls in origin_delay_rate_30d: 115,197

Top 5 worst airports by rolling delay rate:
        origin_delay_rate_30d  ARR_DEL15
ORIGIN                                  
HTS                    0.4417     0.4179
SCK                    0.3754     0.3941
CKB                    0.3579     0.3916
HGR                    0.3726     0.3792
SMX                    0.3787     0.3763


In [71]:
dest_daily = (
    flights.groupby(['DEST', 'FL_DATE'])['ARR_DEL15']
    .mean()
    .reset_index()
    .rename(columns={'ARR_DEL15': 'daily_delay_rate'})
)

dest_daily = dest_daily.sort_values(['DEST', 'FL_DATE'])

dest_daily['dest_delay_rate_30d'] = (
    dest_daily
    .groupby('DEST')['daily_delay_rate']
    .transform(lambda x: x.shift(1).rolling(window=30, min_periods=7).mean())
)

flights = flights.merge(
    dest_daily[['DEST', 'FL_DATE', 'dest_delay_rate_30d']],
    on=['DEST', 'FL_DATE'],
    how='left'
)

print(f"Shape: {flights.shape}")
print(f"Nulls in dest_delay_rate_30d: {flights['dest_delay_rate_30d'].isna().sum():,}")
print(f"\nTop 5 worst dest airports by rolling delay rate:")
print(flights.groupby('DEST')[['dest_delay_rate_30d', 'ARR_DEL15']].mean().round(4).sort_values('ARR_DEL15', ascending=False).head())

Shape: (18227796, 93)
Nulls in dest_delay_rate_30d: 115,210

Top 5 worst dest airports by rolling delay rate:
      dest_delay_rate_30d  ARR_DEL15
DEST                                
PHF                   NaN     0.5000
PIR                0.4625     0.3864
HOB                0.3615     0.3645
PSE                0.3363     0.3370
BQN                0.3329     0.3326


In [72]:
route_daily = (
    flights.groupby(['ORIGIN', 'DEST', 'FL_DATE'])['ARR_DEL15']
    .mean()
    .reset_index()
    .rename(columns={'ARR_DEL15': 'daily_delay_rate'})
)

route_daily = route_daily.sort_values(['ORIGIN', 'DEST', 'FL_DATE'])

route_daily['route_delay_rate_30d'] = (
    route_daily
    .groupby(['ORIGIN', 'DEST'])['daily_delay_rate']
    .transform(lambda x: x.shift(1).rolling(window=30, min_periods=7).mean())
)

flights = flights.merge(
    route_daily[['ORIGIN', 'DEST', 'FL_DATE', 'route_delay_rate_30d']],
    on=['ORIGIN', 'DEST', 'FL_DATE'],
    how='left'
)

print(f"Shape: {flights.shape}")
print(f"Nulls in route_delay_rate_30d: {flights['route_delay_rate_30d'].isna().sum():,}")
print(f"\nTop 5 worst routes by rolling delay rate:")
print(flights.groupby(['ORIGIN','DEST'])[['route_delay_rate_30d','ARR_DEL15']].mean().round(4).sort_values('ARR_DEL15', ascending=False).head())

Shape: (18227796, 94)
Nulls in route_delay_rate_30d: 133,471

Top 5 worst routes by rolling delay rate:
             route_delay_rate_30d  ARR_DEL15
ORIGIN DEST                                 
DAL    DTW                    NaN        1.0
LFT    MSY                    NaN        1.0
TPA    MCO                    NaN        1.0
BUR    FLL                    NaN        1.0
BUF    SEA                    NaN        1.0


In [ ]:
taxi_daily = (
    flights.groupby(['ORIGIN', 'FL_DATE', 'DEP_HOUR'])['TAXI_OUT']
    .mean()
    .reset_index()
    .rename(columns={'TAXI_OUT': 'daily_avg_taxi'})
)

taxi_daily = taxi_daily.sort_values(['ORIGIN', 'DEP_HOUR', 'FL_DATE'])

taxi_daily['origin_avg_taxi_out_30d'] = (
    taxi_daily
    .groupby(['ORIGIN', 'DEP_HOUR'])['daily_avg_taxi']
    .transform(lambda x: x.shift(1).rolling(window=30, min_periods=7).mean())
)

flights = flights.merge(
    taxi_daily[['ORIGIN', 'FL_DATE', 'DEP_HOUR', 'origin_avg_taxi_out_30d']],
    on=['ORIGIN', 'FL_DATE', 'DEP_HOUR'],
    how='left'
)

print(f"Shape: {flights.shape}")
print(f"Nulls: {flights['origin_avg_taxi_out_30d'].isna().sum():,}")
print(f"\nSample — avg taxi at DFW by hour:")
print(flights[flights['ORIGIN']=='DFW'].groupby('DEP_HOUR')['origin_avg_taxi_out_30d'].mean().round(2))

Shape: (18227796, 95)
Nulls: 132,093

Sample — avg taxi at DFW by hour:
DEP_HOUR
0     16.52
1       NaN
5     16.41
6     17.58
7     19.70
8     22.85
9     19.93
10    22.06
11    18.75
12    19.89
13    17.66
14    18.19
15    18.24
16    19.19
17    18.89
18    19.50
19    21.54
20    19.31
21    21.16
22    18.30
23    17.82
Name: origin_avg_taxi_out_30d, dtype: float64


In [ ]:
wind_norm_origin = (flights['origin_wind_speed'] / 30).clip(0, 1)
precip_norm_origin = (flights['origin_precip_1hr'] / 5).clip(0, 1)
sky_norm_origin = flights['origin_sky_condition'] / 9  

flights['origin_weather_severity'] = (
    wind_norm_origin * 0.4 + precip_norm_origin * 0.3 + sky_norm_origin * 0.3
).astype('float32')

wind_norm_dest = (flights['dest_wind_speed'] / 30).clip(0, 1)
precip_norm_dest = (flights['dest_precip_1hr'] / 5).clip(0, 1)
sky_norm_dest = flights['dest_sky_condition'] / 9

flights['dest_weather_severity'] = (
    wind_norm_dest * 0.4 + precip_norm_dest * 0.3 + sky_norm_dest * 0.3
).astype('float32')

print("origin_weather_severity:")
print(f"  Nulls: {flights['origin_weather_severity'].isna().sum():,}")
print(f"  Stats:\n{flights['origin_weather_severity'].describe().round(4)}")

print("\nDelay rate by severity bucket:")
wx_bucket = pd.cut(flights['origin_weather_severity'], bins=[0, 0.1, 0.2, 0.3, 0.5, 1.0])
print(flights.groupby(wx_bucket, observed=True)['ARR_DEL15'].mean() * 100)

origin_weather_severity:
  Nulls: 8,810,842
  Stats:
count    9.416954e+06
mean     1.482000e-01
std      1.029000e-01
min      0.000000e+00
25%      6.670000e-02
50%      1.333000e-01
75%      2.200000e-01
max      7.453000e-01
Name: origin_weather_severity, dtype: float64

Delay rate by severity bucket:
origin_weather_severity
(0.0, 0.1]    15.696107
(0.1, 0.2]    20.198148
(0.2, 0.3]    23.111601
(0.3, 0.5]    25.069795
(0.5, 1.0]    35.228749
Name: ARR_DEL15, dtype: float64


In [75]:
flights.to_parquet("dataset/merged_flights_fe.parquet", 
                   index=False, engine="pyarrow", compression="snappy")
import os
size_mb = os.path.getsize("dataset/merged_flights_fe.parquet") / 1024 / 1024
print(f"✅ Saved. Size: {size_mb:.1f} MB | Shape: {flights.shape}")

✅ Saved. Size: 999.7 MB | Shape: (18227796, 97)


In [ ]:
feature_cols = [c for c in flights.columns if c not in [
    'FL_DATE', 'YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH',
    'OP_UNIQUE_CARRIER', 'OP_CARRIER_AIRLINE_ID', 'TAIL_NUM', 
    'OP_CARRIER_FL_NUM', 'ORIGIN', 'DEST', 'ORIGIN_CITY_NAME', 
    'ORIGIN_STATE_ABR', 'ORIGIN_STATE_NM', 'DEST_CITY_NAME', 
    'DEST_STATE_ABR', 'DEST_STATE_NM', 'AIRLINE_NAME',
    'CRS_DEP_TIME', 'CRS_ARR_TIME', 'CRS_ELAPSED_TIME', 'DEP_DATETIME',

    'ARR_DEL15', 'ARR_DELAY',

    'DEP_TIME', 'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15',
    'ARR_TIME', 'ARR_DELAY_NEW', 'TAXI_OUT', 'TAXI_IN',
    'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'CARRIER_DELAY', 
    'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 
    'LATE_AIRCRAFT_DELAY', 'TAXI_GROUP',
]]

print(f"Total potential model features: {len(feature_cols)}")
for col in feature_cols:
    print(f"  {col}")

Total potential model features: 57
  DAY_OF_WEEK
  DISTANCE
  DEP_HOUR
  DAY_NAME
  DIST_GROUP
  IS_HOLIDAY
  ARR_HOUR
  origin_temp
  origin_dew_point
  origin_pressure
  origin_wind_dir
  origin_wind_speed
  origin_sky_condition
  origin_precip_1hr
  dest_temp
  dest_dew_point
  dest_pressure
  dest_wind_dir
  dest_wind_speed
  dest_sky_condition
  dest_precip_1hr
  OP_REVENUES
  OP_EXPENSES
  NET_INCOME
  OP_PROFIT_LOSS
  FLYING_OPS
  MAINTENANCE
  PAX_SERVICE
  TRANS_REV_PAX
  DEPREC_AMORT
  TRANS_EXPENSES
  origin_monthly_passengers
  origin_monthly_departures
  is_early_departure
  is_peak_hours
  is_high_delay_day
  is_low_delay_day
  is_summer_peak
  is_fall_low
  profit_margin
  maintenance_ratio
  revenue_per_pax
  cost_efficiency
  is_regional
  is_profitable
  origin_is_adverse_sky
  dest_is_adverse_sky
  cascade_score
  cascade_delay_minutes
  hours_since_last_delay
  airline_delay_rate_30d
  origin_delay_rate_30d
  dest_delay_rate_30d
  route_delay_rate_30d
  origin_avg_t

In [77]:
flights.columns

Index(['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE',
       'OP_UNIQUE_CARRIER', 'OP_CARRIER_AIRLINE_ID', 'TAIL_NUM',
       'OP_CARRIER_FL_NUM', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR',
       'ORIGIN_STATE_NM', 'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR',
       'DEST_STATE_NM', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY',
       'DEP_DELAY_NEW', 'DEP_DEL15', 'TAXI_OUT', 'TAXI_IN', 'CRS_ARR_TIME',
       'ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15',
       'CRS_ELAPSED_TIME', 'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'DISTANCE',
       'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY',
       'LATE_AIRCRAFT_DELAY', 'AIRLINE_NAME', 'DEP_HOUR', 'DAY_NAME',
       'DIST_GROUP', 'TAXI_GROUP', 'IS_HOLIDAY', 'ARR_HOUR', 'origin_temp',
       'origin_dew_point', 'origin_pressure', 'origin_wind_dir',
       'origin_wind_speed', 'origin_sky_condition', 'origin_precip_1hr',
       'dest_temp', 'dest_dew_point', 'dest_pressure', 'dest_wind_dir',
     

In [78]:
airline_profile = flights.groupby('OP_UNIQUE_CARRIER').agg(
    airline_name=('AIRLINE_NAME', 'first'),
    flight_count=('ARR_DEL15', 'count'),
    actual_delay_rate=('ARR_DEL15', 'mean'),
    profit_margin=('profit_margin', 'mean'),
    cost_efficiency=('cost_efficiency', 'mean'),
    airline_delay_rate_30d=('airline_delay_rate_30d', 'mean'),
    cascade_score=('cascade_score', 'mean'),
).round(4)

print(airline_profile.sort_values('actual_delay_rate', ascending=False).to_string())

                        airline_name  flight_count  actual_delay_rate  profit_margin  cost_efficiency  airline_delay_rate_30d  cascade_score
OP_UNIQUE_CARRIER                                                                                                                           
F9                          Frontier        504061             0.2991        -0.0068           1.0068                  0.2954         0.2238
B6                           JetBlue        654301             0.2789        -0.0992           1.0992                  0.2796         0.1760
NK                            Spirit        651621             0.2544        -0.1708           1.1708                  0.2538         0.1835
AA                 American Airlines       2520340             0.2514         0.0668           0.9332                  0.2509         0.1643
G4                         Allegiant        319584             0.2427         0.0695           0.9305                  0.2237         0.1927
OH           